# qlib_paper_sim_v1

SOXL qlib-backed production paper/sim 验收卡：固定使用 research/qlib_validated_signal_weight_v1 验证出的 linear_ic 策略，跑 qlib recorder/record，再从 qlib 预测 sidecar 生成 paper target-position intent。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow.record_temp import SignalRecord, SigAnaRecord, PortAnaRecord

root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root.parent == root:
        raise RuntimeError('repo root not found')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.workflow import QlibNativeExperimentSpec, run_qlib_native_experiment

provider_uri = root / 'data' / 'qlib' / 'soxl'
qlib.init(provider_uri=provider_uri.as_posix(), region='us', kernels=1)
_features = D.features(['SOXL'], ['$close'], start_time='2019-01-02', end_time='2026-06-09', freq='day')

card_dir = root / 'production' / 'qlib_paper_sim_v1'
spec = QlibNativeExperimentSpec(
    experiment='qlib_paper_sim_v1',
    provider_uri=provider_uri,
    recorder_uri=card_dir / 'qlib_recorder',
    output_dir=card_dir,
    market='all',
    region='us',
    instrument='SOXL',
    train=('2012-01-03', '2017-12-29'),
    valid=('2018-01-02', '2018-12-31'),
    test=('2019-01-02', '2026-06-09'),
    model_selection_metric='objective_score',
    fixed_model_name='linear_ic',
    promoted_from='research/qlib_validated_signal_weight_v1',
)
result = run_qlib_native_experiment(spec)

predictions = pd.read_pickle(Path(result.summary['recorder_local_dir']) / 'artifacts' / 'pred.pkl')
if hasattr(predictions, 'to_frame'):
    predictions = predictions.to_frame(name='score') if predictions.name == 'score' else predictions
latest_score = float(predictions['score'].iloc[-1])
latest_price = float(_features['$close'].iloc[-1])
threshold = float(result.summary['target_position_strategy']['score_threshold'])
risk_degree = float(result.summary['target_position_strategy']['risk_degree'])
target_percent = risk_degree if latest_score > threshold else 0.0
quantity = int((10_000 * target_percent) / latest_price)
order_intents = pd.DataFrame([
    {
        'symbol': 'SOXL',
        'as_of': spec.test[1],
        'target_percent': target_percent,
        'score': latest_score,
        'score_threshold': threshold,
        'quantity': quantity,
        'price': latest_price,
        'execution_mode': 'paper_sim',
        'real_broker_order': False,
        'recorder_id': result.recorder_id,
    }
])
positions = order_intents[['symbol', 'quantity']]
order_intents.to_csv(card_dir / 'order_intents.csv', index=False)
positions.to_csv(card_dir / 'positions.csv', index=False)
production_summary = {
    'strategy': 'qlib_paper_sim_v1',
    'execution_mode': 'paper_sim',
    'real_broker_orders': False,
    'provider_uri': result.summary['provider_uri'],
    'recorder_id': result.recorder_id,
    'fixed_model_name': 'linear_ic',
    'promoted_from': 'research/qlib_validated_signal_weight_v1',
    'test_window': list(spec.test),
    'strategy_return_pct': result.summary['legacy_soxlbacktest_benchmark']['strategy_return_pct'],
    'legacy_rulebase_return_pct': result.summary['legacy_soxlbacktest_benchmark']['legacy_rulebase_return_pct'],
    'exceeds_legacy_rulebase': result.summary['legacy_soxlbacktest_benchmark']['exceeds_legacy_rulebase'],
    'order_intents': 'order_intents.csv',
    'positions': 'positions.csv',
    'qlib_native_api': result.summary['qlib_native_api'],
}
(card_dir / 'production_summary.json').write_text(json.dumps(production_summary, indent=2, sort_keys=True), encoding='utf-8')
legacy = result.summary['legacy_soxlbacktest_benchmark']
conclusion = f"""# 中文结论

- 做了什么：固定使用 research/qlib_validated_signal_weight_v1 验证出的 linear_ic / qlib_ic_weighted_linear，执行 qlib Dataset、R.start、SignalRecord、SigAnaRecord、PortAnaRecord，并生成 paper/sim target-position intent。
- 固定参数：train=2012-01-03 到 2017-12-29，valid=2018-01-02 到 2018-12-31，test=2019-01-02 到 2026-06-09；model_selection_metric=objective_score；fixed_model_name=linear_ic。
- 核心指标：qlib paper/sim 策略 test 收益 {legacy['strategy_return_pct']:.2f}%，legacy soxl_backtest rulebase test 收益 {legacy['legacy_rulebase_return_pct']:.2f}%，超额 {legacy['excess_return_pct']:.2f}%；recorder_id={result.recorder_id}。
- regime/window 差异：regime_metrics.csv、window_metrics.csv、backtest_window_metrics.csv 和 risk_analysis.csv 为 qlib recorder 派生证据。
- 是否晋升：晋升到 production paper/sim 固定策略；仍不做真实 broker 下单。
- 风险：该卡固定的是验证集选择结果，后续仍需 rolling/walk-forward 复验和 schedule 接入。
- 下一步：让 schedule/dashboard 仅读取 recorder 登记的 production_summary 和 sidecars。
"""
(card_dir / 'conclusion.md').write_text(conclusion, encoding='utf-8')
production_summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[45996:MainThread](2026-06-22 17:38:48,244) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45996:MainThread](2026-06-22 17:38:48,246) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[45996:MainThread](2026-06-22 17:38:48,247) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[45996:MainThread](2026-06-22 17:38:48,254) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[local] 读取 1888 条数据


[45996:MainThread](2026-06-22 17:38:50,979) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45996:MainThread](2026-06-22 17:38:50,980) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[45996:MainThread](2026-06-22 17:38:50,980) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{'strategy': 'qlib_paper_sim_v1',
 'execution_mode': 'paper_sim',
 'real_broker_orders': False,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl',
 'recorder_id': 'e0b5ed66c6514873b2fc79f2aec5874c',
 'fixed_model_name': 'linear_ic',
 'promoted_from': 'research/qlib_validated_signal_weight_v1',
 'test_window': ['2019-01-02', '2026-06-09'],
 'strategy_return_pct': 6570.0232821043,
 'legacy_rulebase_return_pct': 2148.7671608111505,
 'exceeds_legacy_rulebase': True,
 'order_intents': 'order_intents.csv',
 'positions': 'positions.csv',
 'qlib_native_api': ['qlib.init',
  'qlib.data.D.features',
  'DatasetH',
  'DataHandlerLP',
  'Alpha158.get_feature_config',
  'qlib.workflow.R.start',
  'WeightStrategyBase',
  'SignalRecord',
  'SigAnaRecord',
  'PortAnaRecord']}

## 中文结论

- 做了什么：固定使用 validated research 选出的 linear_ic / qlib_ic_weighted_linear，生成 qlib recorder、PortAnaRecord 和 paper/sim target-position intent。
- 核心指标：输出 order_intents.csv、positions.csv、production_summary.json、model_selection.csv、selected_model.json、risk_analysis.csv 和 backtest_benchmark_comparison.csv，并记录 qlib recorder id。
- 产物策略：paper intent 保留为 production 根目录 sidecar；预测、信号、组合明细和全量模型预测保留在 qlib recorder。
- regime 差异：本卡使用 2012-2017 train、2018 valid、2019-2026 test；regime/window 对比由 recorder 派生 sidecar 记录。
- 是否晋升：仅 paper/sim，不做真实 broker 下单。
- 风险：仍是 paper/sim 固定参数验证，不做真实 broker 下单；后续还需要 rolling/walk-forward 复验。
- 下一步：在 schedule 中读取 recorder-registered manifest/export。